In [20]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Dataset, Reader
from scipy.sparse import hstack, csr_matrix

In [95]:
class BookRecommender:
    def __init__(self, books_path='books.csv', ratings_path='ratings.csv',
                 tags_path='tags.csv', book_tags_path='book_tags.csv'):
        self.books_df = pd.read_csv(books_path)
        self.ratings_df = pd.read_csv(ratings_path)
        self.tags_df = pd.read_csv(tags_path)
        self.book_tags_df = pd.read_csv(book_tags_path)
        
        self._fix_book_ids()
        self._prepare_book_features()
        self._build_collaborative_model()

    def _fix_book_ids(self):
        
        local_to_goodreads = dict(zip(
            self.books_df['id'], 
            self.books_df['book_id']
        ))
        
        self.ratings_df.rename(columns={'book_id': 'local_book_id'}, inplace=True)
        
        self.ratings_df['goodreads_book_id'] = self.ratings_df['local_book_id'].map(local_to_goodreads)
        
        self.ratings_df = self.ratings_df.dropna(subset=['goodreads_book_id'])
        self.ratings_df['goodreads_book_id'] = self.ratings_df['goodreads_book_id'].astype(int)
        
        self.book_tags_df.rename(columns={'goodreads_book_id': 'book_id'}, inplace=True)
        

    def _prepare_book_features(self):
        
        book_tags_merged = pd.merge(self.book_tags_df, self.tags_df, on='tag_id')
        book_tags_grouped = book_tags_merged.groupby('book_id')['tag_name'].apply(
            lambda x: ' '.join(x)
        ).reset_index()
        self.books_df = pd.merge(self.books_df, book_tags_grouped, on='book_id', how='left')
        self.books_df['tag_name'].fillna('', inplace=True)

        top_authors = self.books_df['authors'].value_counts().head(50).index
        for author in top_authors:
            self.books_df[f'author_{author}'] = self.books_df['authors'].apply(
                lambda x: 1 if x == author else 0
            )

        self.books_df = self.books_df.dropna(subset=['original_publication_year'])
        year_min = self.books_df['original_publication_year'].min()
        year_max = self.books_df['original_publication_year'].max()
        if year_max > year_min:
            self.books_df['scaled_year'] = (self.books_df['original_publication_year'] - year_min) / (year_max - year_min)
        else:
            self.books_df['scaled_year'] = 0

        self.books_df['clean_title'] = self.books_df['title'].str.replace('[^a-zA-Z ]', '', regex=True)
        self.books_df['text_features'] = self.books_df['clean_title'] + ' ' + self.books_df['tag_name']
        self.text_tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
        text_matrix = self.text_tfidf.fit_transform(self.books_df['text_features'])

        author_cols = [col for col in self.books_df.columns if col.startswith('author_')]
        author_matrix = self.books_df[author_cols].values
        year_matrix = self.books_df['scaled_year'].values.reshape(-1, 1)

        self.book_features_matrix = hstack([text_matrix, author_matrix, year_matrix])
        self.book_features_csr = self.book_features_matrix.tocsr()
        print(f"Создана матрица признаков книг размером: {self.book_features_matrix.shape}")

    def _build_collaborative_model(self):
        reader = Reader(rating_scale=(1, 5))
        data = Dataset.load_from_df(
            self.ratings_df[['user_id', 'goodreads_book_id', 'rating']], 
            reader
        )
        trainset = data.build_full_trainset()
        self.svd_model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02)
        self.svd_model.fit(trainset)

    def get_popular_books(self, n=10):
        popular_books = self.ratings_df.groupby('goodreads_book_id').agg(
            avg_rating=('rating', 'mean'),
            num_ratings=('rating', 'count')
        ).reset_index()
        
        popular_books = popular_books.sort_values(
            ['num_ratings', 'avg_rating'], 
            ascending=[False, False]
        )
        
        top_n_ids = popular_books.head(n)['goodreads_book_id'].tolist()
        
        result = self.books_df[self.books_df['book_id'].isin(top_n_ids)][['book_id', 'title', 'authors']]
        
        if result.empty:
            return pd.DataFrame(columns=['book_id', 'title', 'authors'])
        
        return result

    def get_content_based_recommendations(self, book_ids, n=10):
        book_indices = self.books_df[self.books_df['book_id'].isin(book_ids)].index.tolist()
        if not book_indices:
            return self.get_popular_books(n)
        
        book_vectors = self.book_features_csr[book_indices].toarray()
        user_profile_vector = np.mean(book_vectors, axis=0).reshape(1, -1)
        similarity_scores = cosine_similarity(user_profile_vector, self.book_features_csr).flatten()
        similar_indices = similarity_scores.argsort()[::-1]
        similar_indices = [idx for idx in similar_indices if idx not in book_indices]
        top_indices = similar_indices[:n]
        return self.books_df.iloc[top_indices][['book_id', 'title', 'authors']]

    def get_collaborative_recommendations(self, user_id, n=10):
        if user_id not in self.ratings_df['user_id'].unique():
            return self.get_popular_books(n)
        
        user_rated_books = self.ratings_df[self.ratings_df['user_id'] == user_id]['goodreads_book_id'].tolist()
        all_books = self.books_df['book_id'].tolist()
        books_to_predict = [book_id for book_id in all_books if book_id not in user_rated_books]
        
        if not books_to_predict:
            return self.get_popular_books(n)
        
        predictions = [(book_id, self.svd_model.predict(user_id, book_id).est) for book_id in books_to_predict]
        predictions.sort(key=lambda x: x[1], reverse=True)
        top_n_ids = [book_id for book_id, _ in predictions[:n]]
        result = self.books_df[self.books_df['book_id'].isin(top_n_ids)][['book_id', 'title', 'authors']]
        
        if result.empty:
            return self.get_popular_books(n)
        
        return result

    def get_hybrid_recommendations(self, user_id, n=10):
        if user_id not in self.ratings_df['user_id'].unique():
            return self.get_popular_books(n)
        return self.get_collaborative_recommendations(user_id, n)

In [96]:
BOOKS_PATH = 'books.csv'
RATINGS_PATH = 'ratings.csv'
TAGS_PATH = 'tags.csv'
BOOK_TAGS_PATH = 'book_tags.csv'

recommender = BookRecommender(
    books_path=BOOKS_PATH,
    ratings_path=RATINGS_PATH,
    tags_path=TAGS_PATH,
    book_tags_path=BOOK_TAGS_PATH
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22164\2371079005.py:37: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  self.books_df['tag_name'].fillna('', inplace=True)


Создана матрица признаков книг размером: (9979, 5051)


In [97]:
print("\nГибридные рекомендации для пользователя 1")
recommender.get_hybrid_recommendations(user_id=50, n=5)

   


Гибридные рекомендации для пользователя 1


,book_id,title,authors
1596,122,"The Power of One (The Power of One, #1)",Bryce Courtenay
1787,24813,The Calvin and Hobbes Tenth Anniversary Book,Bill Watterson
4867,481749,Jesus the Christ,James E. Talmage
6919,24815,The Indispensable Calvin and Hobbes,Bill Watterson
8977,121792,The Revenge of the Baby-Sat,Bill Watterson


In [98]:
print("\nГибридные рекомендации для НОВОГО пользователя 99999")
recommender.get_hybrid_recommendations(user_id=99999, n=5)

    


Гибридные рекомендации для НОВОГО пользователя 99999


,book_id,title,authors
3274,8,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...","J.K. Rowling, Mary GrandPré"
5206,24818,The Days Are Just Packed: A Calvin and Hobbes ...,Bill Watterson
5579,24494,The Calvin and Hobbes Lazy Sunday Book,Bill Watterson
6360,70489,There's Treasure Everywhere: A Calvin and Hobb...,Bill Watterson
6919,24815,The Indispensable Calvin and Hobbes,Bill Watterson


In [99]:
print("\nКонтентные рекомендации для книги 'The Hunger Games' (book_id=1)")
recommender.get_content_based_recommendations(book_ids=[1], n=5)


Контентные рекомендации для книги 'The Hunger Games' (book_id=1)


,book_id,title,authors
24,136251,Harry Potter and the Deathly Hallows (Harry Po...,"J.K. Rowling, Mary GrandPré"
17,5,Harry Potter and the Prisoner of Azkaban (Harr...,"J.K. Rowling, Mary GrandPré, Rufus Beck"
22,15881,Harry Potter and the Chamber of Secrets (Harry...,"J.K. Rowling, Mary GrandPré"
23,6,Harry Potter and the Goblet of Fire (Harry Pot...,"J.K. Rowling, Mary GrandPré"
1,3,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré"
